In [1]:
import os
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

project_path = '/content/drive/MyDrive/Spacecraft-Anomaly-Detection'

data_path = project_path + '/data/raw/archive/data/data'
train_path = data_path + '/train'
test_path = data_path + '/test'

print("SERIAL 09 loaded successfully.")
print("Train path:", train_path)
print("Test path:", test_path)

SERIAL 09 loaded successfully.
Train path: /content/drive/MyDrive/Spacecraft-Anomaly-Detection/data/raw/archive/data/data/train
Test path: /content/drive/MyDrive/Spacecraft-Anomaly-Detection/data/raw/archive/data/data/test


In [3]:
from google.colab import drive

drive.mount('/content/drive')

project_path = '/content/drive/MyDrive/Spacecraft-Anomaly-Detection'

data_path = project_path + '/data/raw/archive/data/data'
train_path = data_path + '/train'
test_path = data_path + '/test'

print("Drive mounted successfully.")
print("Train folder exists:", os.path.exists(train_path))
print("Test folder exists:", os.path.exists(test_path))

Mounted at /content/drive
Drive mounted successfully.
Train folder exists: True
Test folder exists: True


In [4]:
# Check missing values across all training channels

missing_summary = []

for file in os.listdir(train_path):
    if file.endswith(".npy"):
        channel = file.replace(".npy", "")
        data = np.load(os.path.join(train_path, file))

        missing_count = np.isnan(data).sum()

        missing_summary.append({
            "Channel": channel,
            "Missing Values": missing_count
        })

missing_df = pd.DataFrame(missing_summary)

print("Total channels checked:", len(missing_df))
print("Channels with missing values:", (missing_df["Missing Values"] > 0).sum())
print("Total missing values:", missing_df["Missing Values"].sum())

Total channels checked: 82
Channels with missing values: 0
Total missing values: 0


In [5]:
# Check infinite values across all training channels

infinite_summary = []

for file in os.listdir(train_path):
    if file.endswith(".npy"):
        channel = file.replace(".npy", "")
        data = np.load(os.path.join(train_path, file))

        infinite_count = np.isinf(data).sum()

        infinite_summary.append({
            "Channel": channel,
            "Infinite Values": infinite_count
        })

infinite_df = pd.DataFrame(infinite_summary)

print("Total channels checked:", len(infinite_df))
print("Channels with infinite values:", (infinite_df["Infinite Values"] > 0).sum())
print("Total infinite values:", infinite_df["Infinite Values"].sum())

Total channels checked: 82
Channels with infinite values: 0
Total infinite values: 0


In [6]:
# Check data types across all training channels

dtype_summary = []

for file in os.listdir(train_path):
    if file.endswith(".npy"):
        channel = file.replace(".npy", "")
        data = np.load(os.path.join(train_path, file))

        dtype_summary.append({
            "Channel": channel,
            "Data Type": str(data.dtype),
            "Numeric": np.issubdtype(data.dtype, np.number)
        })

dtype_df = pd.DataFrame(dtype_summary)

print("Total channels checked:", len(dtype_df))
print("Non-numeric channels:", (~dtype_df["Numeric"]).sum())

print("\nData types found:")
print(dtype_df["Data Type"].value_counts())

Total channels checked: 82
Non-numeric channels: 0

Data types found:
Data Type
float64    82
Name: count, dtype: int64


In [7]:
# Check telemetry value ranges across all training channels

range_summary = []

for file in os.listdir(train_path):
    if file.endswith(".npy"):
        channel = file.replace(".npy", "")
        data = np.load(os.path.join(train_path, file))

        telemetry = data[:, 0]

        range_summary.append({
            "Channel": channel,
            "Minimum": np.min(telemetry),
            "Maximum": np.max(telemetry)
        })

range_df = pd.DataFrame(range_summary)

outside_range = (
    (range_df["Minimum"] < -1) |
    (range_df["Maximum"] > 1)
)

print("Total channels checked:", len(range_df))
print("Channels outside [-1, 1]:", outside_range.sum())
print("Minimum value found:", range_df["Minimum"].min())
print("Maximum value found:", range_df["Maximum"].max())

Total channels checked: 82
Channels outside [-1, 1]: 18
Minimum value found: -1.47721668720188
Maximum value found: 4.162651279553374


In [8]:
# Identify channels with telemetry values outside [-1, 1]

outside_channels = range_df[
    (range_df["Minimum"] < -1) |
    (range_df["Maximum"] > 1)
]

print("Channels outside [-1, 1]:")
print(outside_channels.to_string(index=False))

Channels outside [-1, 1]:
Channel   Minimum  Maximum
   E-13 -1.000000 1.000000
   D-15 -1.000000 1.191578
   D-16 -1.000000 1.008880
    M-4 -1.465464 1.000005
    E-9 -1.000000 1.000000
    M-5 -1.255006 0.981588
    C-1 -1.000000 2.193448
    M-7 -1.002024 0.403249
   P-15  0.730839 1.005220
    M-3 -1.477217 1.000071
    T-8 -1.000000 1.029412
   P-10  0.985882 1.001129
    M-2 -1.210726 0.886325
    F-5 -1.116378 4.162651
    E-4 -1.000000 1.000000
    M-1 -0.916094 2.492298
    F-8 -1.000000 1.130435
   E-12 -1.000000 1.000000


In [9]:
# Count individual telemetry values outside [-1, 1]

total_outside_values = 0
total_telemetry_values = 0

outside_value_summary = []

for file in os.listdir(train_path):
    if file.endswith(".npy"):
        channel = file.replace(".npy", "")
        data = np.load(os.path.join(train_path, file))

        telemetry = data[:, 0]

        outside_count = np.sum(
            (telemetry < -1) | (telemetry > 1)
        )

        total_outside_values += outside_count
        total_telemetry_values += len(telemetry)

        if outside_count > 0:
            outside_value_summary.append({
                "Channel": channel,
                "Outside Values": outside_count,
                "Total Values": len(telemetry)
            })

outside_values_df = pd.DataFrame(outside_value_summary)

print("Total telemetry values checked:", total_telemetry_values)
print("Total values outside [-1, 1]:", total_outside_values)
print("Percentage outside [-1, 1]:",
      (total_outside_values / total_telemetry_values) * 100)

Total telemetry values checked: 196746
Total values outside [-1, 1]: 5113
Percentage outside [-1, 1]: 2.598782186168969


In [10]:
# Show channels with the most values outside [-1, 1]

outside_values_df = outside_values_df.sort_values(
    "Outside Values",
    ascending=False
)

print(outside_values_df.to_string(index=False))

Channel  Outside Values  Total Values
    M-3            1588          2037
   D-15            1009          2074
    M-2             769          2208
   E-12             595          2880
    M-7             391          1587
    M-4             329          2076
    F-5             228          2598
    M-1              71          2209
   P-15              40          3682
    M-5              36          2032
    E-9              16          2880
    F-8              15          3342
   D-16              11          1451
    C-1              10          2158
    T-8               2           748
   E-13               1          2880
   P-10               1          4308
    E-4               1          2880


In [11]:
# Check telemetry values outside [-1, 1] in the test dataset

test_outside_values = 0
test_total_values = 0

for file in os.listdir(test_path):
    if file.endswith(".npy"):
        data = np.load(os.path.join(test_path, file))
        telemetry = data[:, 0]

        test_outside_values += np.sum(
            (telemetry < -1) | (telemetry > 1)
        )

        test_total_values += len(telemetry)

print("Total test telemetry values checked:", test_total_values)
print("Test values outside [-1, 1]:", test_outside_values)
print(
    "Percentage outside [-1, 1]:",
    (test_outside_values / test_total_values) * 100
)

Total test telemetry values checked: 510225
Test values outside [-1, 1]: 5557
Percentage outside [-1, 1]: 1.089127345778823


In [12]:
# Check consecutive duplicate telemetry values

duplicate_summary = []

for file in os.listdir(train_path):
    if file.endswith(".npy"):
        channel = file.replace(".npy", "")
        data = np.load(os.path.join(train_path, file))

        telemetry = data[:, 0]

        duplicate_count = np.sum(telemetry[1:] == telemetry[:-1])

        duplicate_summary.append({
            "Channel": channel,
            "Consecutive Duplicates": duplicate_count,
            "Total Values": len(telemetry)
        })

duplicate_df = pd.DataFrame(duplicate_summary)

print("Total channels checked:", len(duplicate_df))
print(
    "Channels with consecutive duplicates:",
    (duplicate_df["Consecutive Duplicates"] > 0).sum()
)
print(
    "Total consecutive duplicate pairs:",
    duplicate_df["Consecutive Duplicates"].sum()
)

Total channels checked: 82
Channels with consecutive duplicates: 74
Total consecutive duplicate pairs: 114268


In [13]:
# Identify channels with constant telemetry signals

constant_channels = []

for file in os.listdir(train_path):
    if file.endswith(".npy"):
        channel = file.replace(".npy", "")
        data = np.load(os.path.join(train_path, file))

        telemetry = data[:, 0]

        if np.std(telemetry) == 0:
            constant_channels.append(channel)

print("Total channels checked:", 82)
print("Constant telemetry channels:", len(constant_channels))
print("\nChannels:")
print(constant_channels)

Total channels checked: 82
Constant telemetry channels: 14

Channels:
['B-1', 'D-7', 'D-2', 'D-8', 'D-13', 'C-2', 'D-9', 'D-12', 'P-4', 'M-6', 'G-2', 'S-2', 'D-14', 'T-5']


In [14]:
# Calculate percentage of constant telemetry channels

constant_percentage = (len(constant_channels) / 82) * 100

print("Constant channels:", len(constant_channels))
print("Total channels:", 82)
print("Percentage of constant channels:", constant_percentage)

Constant channels: 14
Total channels: 82
Percentage of constant channels: 17.073170731707318


In [15]:
# Create overall data-cleaning summary

cleaning_summary = {
    "Total Channels": 82,
    "Channels with Missing Values": int(
        (missing_df["Missing Values"] > 0).sum()
    ),
    "Total Missing Values": int(
        missing_df["Missing Values"].sum()
    ),
    "Channels with Infinite Values": int(
        (infinite_df["Infinite Values"] > 0).sum()
    ),
    "Total Infinite Values": int(
        infinite_df["Infinite Values"].sum()
    ),
    "Non-Numeric Channels": int(
        (~dtype_df["Numeric"]).sum()
    ),
    "Train Values Outside [-1,1]": int(total_outside_values),
    "Test Values Outside [-1,1]": int(test_outside_values),
    "Constant Telemetry Channels": len(constant_channels),
    "Consecutive Duplicate Pairs": int(
        duplicate_df["Consecutive Duplicates"].sum()
    )
}

cleaning_summary_df = pd.DataFrame(
    [cleaning_summary]
)

print(cleaning_summary_df.T)

                                    0
Total Channels                     82
Channels with Missing Values        0
Total Missing Values                0
Channels with Infinite Values       0
Total Infinite Values               0
Non-Numeric Channels                0
Train Values Outside [-1,1]      5113
Test Values Outside [-1,1]       5557
Constant Telemetry Channels        14
Consecutive Duplicate Pairs    114268


In [16]:
# Save cleaning summary

summary_path = project_path + '/results/cleaning_summary.csv'

cleaning_summary_df.to_csv(
    summary_path,
    index=False
)

print("Cleaning summary saved successfully.")
print("Saved to:", summary_path)

Cleaning summary saved successfully.
Saved to: /content/drive/MyDrive/Spacecraft-Anomaly-Detection/results/cleaning_summary.csv


In [17]:
# Verify the cleaning summary file

print("File exists:", os.path.exists(summary_path))

if os.path.exists(summary_path):
    print("File size:", os.path.getsize(summary_path), "bytes")
    print("\nSaved summary:")
    print(pd.read_csv(summary_path))

File exists: True
File size: 286 bytes

Saved summary:
   Total Channels  Channels with Missing Values  Total Missing Values  \
0              82                             0                     0   

   Channels with Infinite Values  Total Infinite Values  Non-Numeric Channels  \
0                              0                      0                     0   

   Train Values Outside [-1,1]  Test Values Outside [-1,1]  \
0                         5113                        5557   

   Constant Telemetry Channels  Consecutive Duplicate Pairs  
0                           14                       114268  


#github push

In [18]:
%cd /content/drive/MyDrive/Spacecraft-Anomaly-Detection

!git status

/content/drive/MyDrive/Spacecraft-Anomaly-Detection
Refresh index: 100% (10/10), done.
On branch main
Your branch is up to date with 'origin/main'.

Changes not staged for commit:
  (use "git add <file>..." to update what will be committed)
  (use "git restore <file>..." to discard changes in working directory)
	modified:   notebooks/07_understand_dataset.ipynb
	modified:   notebooks/08_data_visualization.ipynb

Untracked files:
  (use "git add <file>..." to include in what will be committed)
	notebooks/09_data_cleaning.ipynb
	results/cleaning_summary.csv

no changes added to commit (use "git add" and/or "git commit -a")


In [19]:
%cd /content/drive/MyDrive/Spacecraft-Anomaly-Detection

!git add notebooks/07_understand_dataset.ipynb
!git add notebooks/08_data_visualization.ipynb
!git add notebooks/09_data_cleaning.ipynb
!git add results/cleaning_summary.csv

!git commit -m "Complete SERIAL 09 data cleaning"

/content/drive/MyDrive/Spacecraft-Anomaly-Detection
Author identity unknown

*** Please tell me who you are.

Run

  git config --global user.email "you@example.com"
  git config --global user.name "Your Name"

to set your account's default identity.
Omit --global to set the identity only in this repository.

fatal: unable to auto-detect email address (got 'root@f9f46d2d2b26.(none)')


In [20]:
!git config --global user.name "Amit Chandra Das"
!git config --global user.email "arickroy0@gmail.com"

print("Git identity configured.")

Git identity configured.


In [21]:
%cd /content/drive/MyDrive/Spacecraft-Anomaly-Detection

!git commit -m "Complete SERIAL 09 data cleaning"

/content/drive/MyDrive/Spacecraft-Anomaly-Detection
[main 36567ae] Complete SERIAL 09 data cleaning
 4 files changed, 5 insertions(+), 2 deletions(-)
 create mode 100644 notebooks/09_data_cleaning.ipynb
 create mode 100644 results/cleaning_summary.csv


In [22]:
%cd /content/drive/MyDrive/Spacecraft-Anomaly-Detection

!git push origin main

/content/drive/MyDrive/Spacecraft-Anomaly-Detection
Enumerating objects: 13, done.
Counting objects: 100% (13/13), done.
Delta compression using up to 2 threads
Compressing objects: 100% (8/8), done.
Writing objects: 100% (8/8), 7.01 KiB | 718.00 KiB/s, done.
Total 8 (delta 4), reused 0 (delta 0), pack-reused 0
remote: Resolving deltas: 100% (4/4), completed with 4 local objects.
To https://github.com/Amit-Chandra-Das/spacecraft-anomaly-detection.git
   ea80331..36567ae  main -> main
